# SWAP-Stress: Earth Engine covariates (a toy tour around one ReESH site)

Stage 01 (`swapstress-extract`) samples a stack of Earth Engine covariates at
every training site and folds the exports into per-source feature tables. This
notebook opens that stack up around a single station so you can see what is in
it.

1. Where stage 01 reads and writes (dry run — no Earth Engine call)
2. Loading a ReESH station point from the registry's site shapefile
3. Building the covariate stack with `swapstress.features.call_ee`
4. What the bands are called and which group each belongs to
5. Sampling and visualizing a few layers in a ~4 km neighborhood

**Earth Engine costs quota.** `RUN_EE` is `False` by default; sections 3 onward
do nothing until you set it to `True` and have credentials. Section 1 works
either way.

Sentinel-2 is **not** in the covariate stack. The last section shows it purely as
a visual aid and is labeled as such.

In [ ]:
from __future__ import annotations

import os
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from swapstress.sources.registry import DataPaths, get_source

# ---------------------------------------------------------------------------
# The one path you may have to change: where the project data tree is mounted.
# The site shapefile and the MGRS tile index resolve from it through the
# registry; nothing else in this notebook is a literal path.
# ---------------------------------------------------------------------------
DATA_ROOT = os.environ.get("SWAPSTRESS_DATA_ROOT", "/nas/soils")

# Earth Engine calls cost quota. Nothing below section 2 runs while this is False.
RUN_EE = False
SHOW_SENTINEL2_QUICKLOOK = True  # visualization only; NOT in the covariate stack

SITE_ID = "US-CDM"  # a ReESH station
BUFFER_M = 4000
RESOLUTION_M = 250  # the sampling scale stage 01 uses
IMAGE_SIZE = 768

OUT_DIR = os.path.join("notebooks", "_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("SITE_ID:  ", SITE_ID)
print("RUN_EE:   ", RUN_EE)

## 1) What stage 01 does

The Earth Engine export starts one batch task per MGRS tile writing CSVs to Cloud
Storage. Those have to be synced down before they can be folded into feature
tables, and the wait is manual, so the stage splits in two:

```bash
uv run swapstress-extract --step export --data-root /nas/soils   # submits the EE tasks
# ... wait, then sync gs://<bucket>/swapstress/... down to the local extracts dir
uv run swapstress-extract --step tables --data-root /nas/soils   # builds the parquets
```

The dry run below resolves the sites shapefile, the MGRS index, the Cloud Storage
prefix, the local extract directory, and the output feature table for every
source. It does not authenticate and does not call Earth Engine.

In [ ]:
from swapstress.features import ee_export

ee_export.main(["--data-root", DATA_ROOT, "--dry-run"])

## 2) A ReESH station point

The site geometry comes from the registry — `DataPaths(...).shapefile` — so this
follows whatever the registry currently points at rather than a path pasted into
the notebook.

The shapefile writes site ids with underscores (`US_CDM`) while the standardized
observation files use hyphens (`US-CDM`).

In [ ]:
import geopandas as gpd

reesh_paths = DataPaths(DATA_ROOT, get_source("reesh"))
reesh_shp = reesh_paths.shapefile

sites = gpd.read_file(reesh_shp)
sites = sites.to_crs(4326) if sites.crs else sites.set_crs(4326)

row = sites.loc[sites["site_id"].astype(str) == SITE_ID.replace("-", "_")]
if row.empty:
    raise KeyError(f"site_id={SITE_ID} not found in {reesh_shp}")

geom = row.geometry.iloc[0]
site_lon_lat = (float(geom.x), float(geom.y))

print("sites shapefile:", reesh_shp)
print("MGRS index:     ", reesh_paths.mgrs_shapefile)
print("site_lon_lat:   ", site_lon_lat)

## 3) The covariate stack

`swapstress.features.call_ee.stack_bands_climatology` builds the stack stage 01
samples. `region="conus"` adds the CONUS-only layers (PRISM, SSURGO, POLARIS,
NLCD, CDL) on top of the global ones; `region="global"` is what the released
model uses.

In [ ]:
stack = None
bands = []

if not RUN_EE:
    print("RUN_EE is False; skipping every Earth Engine call in this notebook")
else:
    import ee

    from swapstress.features.call_ee import is_authorized, stack_bands_climatology
    from swapstress.features.ee_feature_list import label_feature

    is_authorized()

    lon, lat = site_lon_lat
    point = ee.Geometry.Point([lon, lat])
    roi = point.buffer(BUFFER_M)
    region = roi.bounds()

    stack = stack_bands_climatology(roi, region="conus")
    bands = stack.bandNames().getInfo()

    print(f"Band count: {len(bands)}\n")
    for b in bands[:40]:
        print(f"  {b:<32s} {label_feature(b)}")

### 3a) Which group each band belongs to

`swapstress.features.features.classify_feature` is the same grouping the model
uses for feature selection and for the group-ablation analysis, so these counts
are the ones a `feature_groups` entry in a train config selects over.
`FEATURE_GROUPS` is the full catalogue of group names.

In [ ]:
from swapstress.features.features import FEATURE_GROUPS, classify_feature

print("known feature groups:", ", ".join(sorted(FEATURE_GROUPS)))

if bands:
    counts = pd.Series([classify_feature(b) for b in bands]).value_counts()
    print()
    print(counts.to_string())

### 3b) Sample the stack at the station

A join sanity check: does the stack return non-null values at the station
location? The full stack is too heavy to sample at once, so this reduces a
handful of bands.

In [ ]:
if stack is not None:
    wanted = ["nd_mean_gs", "elevation", "VV_mean", "clay_0-5cm_mean", "slope"]
    sample_bands = [b for b in wanted if b in bands] or bands[:5]

    preview = {}
    for b in sample_bands:
        value = (
            stack.select([b])
            .reduceRegion(
                reducer=ee.Reducer.first(), geometry=point, scale=RESOLUTION_M
            )
            .getInfo()
            .get(b)
        )
        preview[b] = round(value, 4) if isinstance(value, float) else value

    print(f"Sampled {len(preview)} bands at {SITE_ID}:")
    for k, v in preview.items():
        print(f"  {k:<24s} {v}")

## 4) Neighborhood thumbnails

Build an image, `.visualize(...)`, `.getThumbURL(...)`, download, display. The
helper below is notebook display plumbing — it has no pipeline equivalent and
computes nothing.

In [ ]:
def show_thumb(url: str, title: str | None = None):
    """Download an Earth Engine thumbnail URL and draw it."""
    import requests
    from PIL import Image

    response = requests.get(url, timeout=60)
    response.raise_for_status()
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(np.array(Image.open(BytesIO(response.content))))
    ax.axis("off")
    if title:
        ax.set_title(title)
    plt.show()

In [ ]:
if stack is not None:

    def find_band(*candidates):
        for c in candidates:
            if c in bands:
                return c
        for c in candidates:
            for b in bands:
                if c in b:
                    return b
        return None

    picks = [
        (
            find_band("nd_mean_gs", "nd_"),
            "NDVI composite (Landsat)",
            -0.1,
            0.9,
            ["#2c7bb6", "#ffffbf", "#d7191c"],
        ),
        (
            find_band("VV_mean", "VV_"),
            "Sentinel-1 VV mean",
            -25,
            0,
            ["black", "#4a90e2", "white"],
        ),
        (
            find_band("elevation", "elev"),
            "Elevation",
            0,
            3000,
            ["#1a9850", "#fee08b", "#d73027"],
        ),
    ]

    for band, title, vmin, vmax, palette in picks:
        if band is None:
            print(f"skipping {title}: no matching band in the stack")
            continue
        url = (
            stack.select([band])
            .visualize(min=vmin, max=vmax, palette=palette)
            .getThumbURL({"region": region, "dimensions": IMAGE_SIZE})
        )
        print(f"{title}  [{band}]")
        show_thumb(url)

## 5) One band as an array

`sampleRectangle` pulls a small raster block back into the notebook. This is a
sanity check and a visualization aid, not a production export — stage 01 exports
point samples to Cloud Storage, and
`swapstress.features.ee_export_conus_rasters` is what exports gridded covariates.

In [ ]:
if stack is not None:
    band = find_band("nd_mean_gs", "nd_")
    if band is None:
        print("no NDVI-like band in the stack; skipping")
    else:
        props = (
            stack.select([band])
            .reproject(crs="EPSG:4326", scale=RESOLUTION_M)
            .sampleRectangle(region=region, defaultValue=-9999)
            .getInfo()
            .get("properties", {})
        )
        arr = np.array(props.get(band, []), dtype=float)
        arr[arr == -9999] = np.nan

        fig, ax = plt.subplots(figsize=(6, 6))
        im = ax.imshow(arr, cmap="RdYlBu_r", vmin=-0.1, vmax=0.9)
        ax.set_title(f"{band} (sampleRectangle @ {RESOLUTION_M} m)")
        fig.colorbar(im, ax=ax, shrink=0.8)
        out = os.path.join(OUT_DIR, "earth_engine_ndvi_grid.png")
        fig.savefig(out, dpi=200)
        plt.show()
        print("Saved:", os.path.abspath(out))

## 6) Sentinel-2 quicklook (visualization only)

Sentinel-2 is **not** part of the SWAP-Stress covariate stack — it does not
appear in `stack_bands_climatology` and no model feature derives from it. This
cell is here for intuition about what the neighborhood looks like, nothing more.

In [ ]:
if stack is not None and SHOW_SENTINEL2_QUICKLOOK:
    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate("2023-06-01", "2023-09-30")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .median()
        .select(["B4", "B3", "B2"])
    )
    url = s2.visualize(min=0, max=3000).getThumbURL(
        {"region": region, "dimensions": IMAGE_SIZE}
    )
    show_thumb(url, title=f"Sentinel-2 RGB near {SITE_ID} (not a model input)")
else:
    print("Skipping Sentinel-2 quicklook")

## Next

`03_amsr_vod.ipynb` covers the one covariate that is not sampled through Earth
Engine, and `04_training_table.ipynb` joins the sampled covariates to the
observations (stage 02).